In [1]:
import sys
print(sys.executable)

C:\Users\ajant\anaconda3\envs\mediapipe\python.exe


In [6]:
import cv2
import mediapipe as mp

# -----------------------------------
# GitHub Source Link
# -----------------------------------
github_link = "https://github.com/ajanthadevi2012"

# -----------------------------------
# MediaPipe Hands Setup
# -----------------------------------
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=10,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

# -----------------------------------
# Open Webcam
# -----------------------------------
cap = cv2.VideoCapture(0)

# Get camera properties
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

if fps == 0:
    fps = 30

# -----------------------------------
# Output Video
# -----------------------------------
output_path = "gesture_control_output.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)

# -----------------------------------
# Check if fingers are closed
# -----------------------------------
def fingers_closed(lm):

    return sum(
        lm[tip].y > lm[pip].y
        for tip, pip in [
            (8, 6),
            (12, 10),
            (16, 14),
            (20, 18)
        ]
    ) >= 4


# -----------------------------------
# Main Loop
# -----------------------------------
while True:

    ret, frame = cap.read()

    if not ret:
        break

    # Mirror view
    frame = cv2.flip(frame, 1)

    # Convert BGR to RGB
    rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    # Process frame
    result = hands.process(rgb)

    gesture = "No Hands"

    # Detect hands
    if result.multi_hand_landmarks:

        hand = result.multi_hand_landmarks[0]

        # Detect gesture
        gesture = (
            "FIST"
            if fingers_closed(hand.landmark)
            else "Hands are Open"
        )

        # Draw landmarks
        mp_draw.draw_landmarks(
            frame,
            hand,
            mp_hands.HAND_CONNECTIONS
        )

    # -----------------------------------
    # Display Gesture
    # -----------------------------------
    cv2.putText(
        frame,
        gesture,
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (255, 0, 0),
        2
    )

    # -----------------------------------
    # Display GitHub Link - Top Right
    # -----------------------------------
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.45
    thickness = 1

    (text_width, text_height), _ = cv2.getTextSize(
        github_link,
        font,
        font_scale,
        thickness
    )

    x = width - text_width - 10
    y = 25

    cv2.putText(
        frame,
        github_link,
        (x, y),
        font,
        font_scale,
        (255, 255, 255),
        thickness
    )

    # -----------------------------------
    # Save Processed Frame
    # -----------------------------------
    out.write(frame)

    # Display Output
    cv2.imshow("Gesture Control", frame)

    # Press Q to stop
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# -----------------------------------
# Release Resources
# -----------------------------------
cap.release()
out.release()
hands.close()
cv2.destroyAllWindows()

print("Video saved as:", output_path)

Video saved as: gesture_control_output.mp4
